In [1]:
import pandas as pd
from pathlib import Path

In [2]:
interim_df = pd.read_parquet("../data/interim/lastfm/lastfm_scrobbles_merged.parquet")

In [3]:
interim_df.shape

(559449, 4)

In [4]:
interim_df.head()

,artist,track,timestamp,year_file
0,Oasis,Don't Look Back in Anger,1199052426,2007
1,Oasis,Wonderwall,1199052167,2007
2,Oasis,Don't Look Back in Anger,1199051867,2007
3,Oasis,Wonderwall,1199051609,2007
4,Incubus,Aqueous Transmission,1199051161,2007


In [5]:
interim_df["artist"].value_counts().head(20)

artist
The National             9614
Radiohead                8275
Placebo                  6581
Interpol                 5551
Foo Fighters             4775
The Smiths               4568
Foals                    4263
Incubus                  4190
Massive Attack           4161
Japanese Breakfast       4089
Arctic Monkeys           3733
Mitski                   3690
Red Hot Chili Peppers    3593
The Cure                 3428
PJ Harvey                3416
Bat for Lashes           3310
Arcade Fire              3155
White Lies               2783
Cat Power                2753
Coldplay                 2718
Name: count, dtype: int64

In [ ]:
# 1. Drop rows with missing timestamps
df = interim_df.dropna(subset=["timestamp"])
df

In [ ]:
# 2. Convert timestamps to datetime and extract year and month
df["timestamp"] = pd.to_numeric(df["timestamp"], errors="coerce")
df["timestamp"] = df["timestamp"].astype(int)
df["played_at"] = pd.to_datetime(df["timestamp"], unit="s")
df["year"] = df["played_at"].dt.year
df["month"] = df["played_at"].dt.to_period("M")
df

In [ ]:
# 4. Sanity check
print(f"{df.shape} records after processing.")
print(f"Number of empty rows {df.isna().sum()}")
print(f"Number of unique artists in the df {df.artist.nunique()}")
print(f"Number of unique tracks in the df {df.track.nunique()}")
print(f"Top rows {df.head()}")

In [ ]:
# 5. Deduplication
df.duplicated(subset=["artist", "track", "timestamp"]).sum()
df = df.drop_duplicates(subset=["artist", "track", "timestamp"])
df

In [62]:
# 6. Cleaning artist names
import re

def clean_text_artist(text):
    if text is None:
        return None
    
    text = text.lower()
    
    # remove brackets (artist rarely has any important info in brackets)
    text = re.sub(r"\(.*?\)", "", text)
    
    # remove strange suffixes
    text = re.sub(r"\d{2,}[a-z]{3,}\d*$", "", text)
    
    # remove " - something"
    text = re.sub(r"\s-\s.*", "", text)

    # remove special characters
    text = re.sub(r"[^a-z0-9\s]", "", text)

    # normalize whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text if text else None

In [63]:
# 7. Cleaning track names

def clean_text_track(text):
    if text is None:
        return None
    
    text = text.lower()
    
    # Remove content in parentheses that is not part of the title
    text = re.sub(
        r"\((?:"
        r"feat\.?.*?|ft\.?.*?|"
        r"remaster(?:ed)?(?: \d{4})?|"
        r"live(?: at .*?)?|"
        r"radio edit|edit|"
        r"(?:acoustic|alternative|instrumental)(?: version)?|"
        r"mix|version|"
        r"deluxe|bonus track|"
        r"single version"
        r")\)",
        "",
        text,
    )
    
    # remove feat/ft also if outside brackets
    text = re.sub(r"\b(feat|ft)\.?\b.*", "", text)
    
    # 🔥 remove remix/rework/edit/version also if outside brackets
    text = re.sub(r"\b(remix|rework|edit|version)\b.*", "", text)
    
    # remove strange suffixes
    text = re.sub(r"\d{2,}[a-z]{3,}\d*$", "", text)
    
    # remove " - something"
    text = re.sub(r"\s-\s.*", "", text)

    # remove special characters
    text = re.sub(r"[^a-z0-9\s]", "", text)

    # normalize whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text if text else None

In [64]:
df["artist_clean"] = df["artist"].apply(clean_text_artist)
df["track_clean"] = df["track"].apply(clean_text_track)

In [68]:
df = df.dropna(subset=["track_clean"])

In [ ]:
print("Before:", df["track"].nunique())
print("After:", df["track_clean"].nunique())

print("Before:", df["artist"].nunique())
print("After:", df["artist_clean"].nunique())

In [ ]:
df[["track", "track_clean"]].sample(50)

In [ ]:
df[["artist", "artist_clean"]].sample(50)

In [49]:
df["parentheses"] = df["track"].apply(lambda x: re.findall(r"\(.*?\)", x.lower()))

In [ ]:
df["parentheses"].explode().value_counts().head(30)

In [74]:
df.to_parquet(
    "../data/processed/lastfm_scrobbles_clean.parquet",
    index=False
)